In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

import sys
sys.path.append("../../../utils/")

from utils import *

In [2]:
# ===== RUTAS =====
PROJECT_ROOT = Path.cwd().resolve().parents[3]

NOMBRE_DATASET_RESULTADO = "CIC18__cleanning__v2"

# Ajusta esto si hace falta
RUTA_BASE_RAW = PROJECT_ROOT / "02_datasets" / "raw" / "CIC18"
RUTA_SALIDA = PROJECT_ROOT / "02_datasets" / "processed_analisis_estadistico" / NOMBRE_DATASET_RESULTADO

NOMBRE_DATASET_LIMPIO = f"{NOMBRE_DATASET_RESULTADO}.csv"
NOMBRE_REPORTE = f"{NOMBRE_DATASET_RESULTADO}_report.json"

# ===== PARÁMETROS =====
LABEL_COL = "LABEL"
CORR_THRESHOLD = 0.95
MIN_CLASS_IMPUTE = 50

# ===== COLUMNAS A ELIMINAR MANUALMENTE SI EXISTEN =====
COLUMNAS_A_ELIMINAR = [
    "FLOW_ID",
    "SRC_IP",
    "SRC_PORT",
    "DST_IP",
    "TIMESTAMP"
]

In [3]:
print("Carpeta actual del notebook:")
print(PROJECT_ROOT)
print()
print("Ruta base raw:")
print(Path(RUTA_BASE_RAW).resolve())
print()
print("Ruta salida:")
print(Path(RUTA_SALIDA).resolve())
print()

Carpeta actual del notebook:
/LUSTRE/home/inginf/u32902122/TFG

Ruta base raw:
/LUSTRE/home/inginf/u32902122/TFG/02_datasets/raw/CIC18

Ruta salida:
/LUSTRE/home/inginf/u32902122/TFG/02_datasets/processed_analisis_estadistico/CIC18__cleanning__v2



In [4]:
df = cargar_dataset(
    ruta_base=RUTA_BASE_RAW
)

shape_original = df.shape

print("Forma original del dataset:")
print(shape_original)

df.head()

/LUSTRE/home/inginf/u32902122/TFG/03_codigo/analisis_estadistico/CIC18/01_clean_dataset/../../../utils/utils.py:56: ParserWarning: Skipping line 766748: expected 80 fields, saw 103

  df_temp = pd.read_csv(


Forma original del dataset:
(16232952, 84)


,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,...,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,Flow ID,Src IP,Src Port,Dst IP
0,443,6,02/03/2018 08:47:38,141385,9,7,553,3773.0,202,0,...,0.0,0.0,0.0,0.0,0.0,Benign,NaN,NaN,NaN,NaN
1,49684,6,02/03/2018 08:47:38,281,2,1,38,0.0,38,0,...,0.0,0.0,0.0,0.0,0.0,Benign,NaN,NaN,NaN,NaN
2,443,6,02/03/2018 08:47:40,279824,11,15,1086,10527.0,385,0,...,0.0,0.0,0.0,0.0,0.0,Benign,NaN,NaN,NaN,NaN
3,443,6,02/03/2018 08:47:40,132,2,0,0,0.0,0,0,...,0.0,0.0,0.0,0.0,0.0,Benign,NaN,NaN,NaN,NaN
4,443,6,02/03/2018 08:47:41,274016,9,13,1285,6141.0,517,0,...,0.0,0.0,0.0,0.0,0.0,Benign,NaN,NaN,NaN,NaN


In [5]:
df = homogeneizar_columnas(df)

print("Primeras columnas tras homogeneización:")
print(df.columns.tolist()[:20])

Primeras columnas tras homogeneización:
['DST_PORT', 'PROTOCOL', 'TIMESTAMP', 'FLOW_DURATION', 'TOT_FWD_PKTS', 'TOT_BWD_PKTS', 'TOTLEN_FWD_PKTS', 'TOTLEN_BWD_PKTS', 'FWD_PKT_LEN_MAX', 'FWD_PKT_LEN_MIN', 'FWD_PKT_LEN_MEAN', 'FWD_PKT_LEN_STD', 'BWD_PKT_LEN_MAX', 'BWD_PKT_LEN_MIN', 'BWD_PKT_LEN_MEAN', 'BWD_PKT_LEN_STD', 'FLOW_BYTS_S', 'FLOW_PKTS_S', 'FLOW_IAT_MEAN', 'FLOW_IAT_STD']


In [6]:
if LABEL_COL not in df.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL}")

print("Columna objetivo encontrada correctamente.")
print()
print("Distribución inicial de clases:")
display(df[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Columna objetivo encontrada correctamente.

Distribución inicial de clases:


,count
LABEL,
Benign,13484658
DDOS attack-HOIC,686012
DDoS attacks-LOIC-HTTP,576191
DoS attacks-Hulk,461912
Bot,286191
FTP-BruteForce,193360
SSH-Bruteforce,187589
Infilteration,161934
DoS attacks-SlowHTTPTest,139890


In [7]:
columnas_presentes_para_eliminar = [c for c in COLUMNAS_A_ELIMINAR if c in df.columns]

df = eliminar_columnas(df, columnas_presentes_para_eliminar)

print("Columnas eliminadas manualmente:")
print(columnas_presentes_para_eliminar)
print()
print("Forma actual:")
print(df.shape)

Columnas eliminadas manualmente:
['FLOW_ID', 'SRC_IP', 'SRC_PORT', 'DST_IP', 'TIMESTAMP']

Forma actual:
(16232952, 79)


In [8]:
df = limpiar_infinitos_y_vacios(df)

print("Limpieza de infinitos y vacíos completada.")

Limpieza de infinitos y vacíos completada.


In [9]:
filas_antes_duplicados = len(df)

df = eliminar_filas_duplicadas(df)

filas_despues_duplicados = len(df)
duplicados_eliminados = filas_antes_duplicados - filas_despues_duplicados

print("Duplicados eliminados:", duplicados_eliminados)
print("Forma actual:", df.shape)

Duplicados eliminados: 4178920
Forma actual: (12054032, 79)


In [10]:
df_sin_nulos, df_con_nulos = separar_filas_con_y_sin_nulos(df)

print("Filas sin nulos:", len(df_sin_nulos))
print("Filas con nulos:", len(df_con_nulos))

Filas sin nulos: 11982426
Filas con nulos: 71606


In [11]:
df, filas_imputadas, filas_eliminadas_nulos = imputar_o_eliminar_nulos_por_clase(
    df_sin_nulos=df_sin_nulos,
    df_con_nulos=df_con_nulos,
    label_col=LABEL_COL,
    min_class_impute=MIN_CLASS_IMPUTE
)

print("Filas imputadas:", filas_imputadas)
print("Filas eliminadas por nulos:", filas_eliminadas_nulos)
print("Forma actual:", df.shape)

Filas imputadas: 0
Filas eliminadas por nulos: 71606
Forma actual: (11982426, 79)


In [12]:
columnas_antes_constantes = df.shape[1]

df = eliminar_columnas_constantes(df)

columnas_despues_constantes = df.shape[1]
constantes_eliminadas = columnas_antes_constantes - columnas_despues_constantes

print("Columnas constantes eliminadas:", constantes_eliminadas)
print("Forma actual:", df.shape)

Columnas constantes eliminadas: 0
Forma actual: (11982426, 79)


In [13]:
df = codificar_columnas_string(df)

In [14]:
columnas_antes_corr = df.shape[1]

df = eliminar_columnas_altamente_correlacionadas(
    df,
    threshold=CORR_THRESHOLD,
    label_col=LABEL_COL
)

columnas_despues_corr = df.shape[1]
corr_eliminadas = columnas_antes_corr - columnas_despues_corr

print("Columnas eliminadas por alta correlación:", corr_eliminadas)
print("Forma final tras limpieza:", df.shape)

Columnas eliminadas por alta correlación: 24
Forma final tras limpieza: (11982426, 55)


In [15]:
feature_cols = [c for c in df.columns if c != LABEL_COL]
df = df[feature_cols + [LABEL_COL]]

print("Última columna:", df.columns[-1])
df.head()

Última columna: LABEL


,DST_PORT,PROTOCOL,FLOW_DURATION,TOT_FWD_PKTS,TOT_BWD_PKTS,TOTLEN_FWD_PKTS,TOTLEN_BWD_PKTS,FWD_PKT_LEN_MAX,FWD_PKT_LEN_MIN,FWD_PKT_LEN_MEAN,...,INIT_BWD_WIN_BYTS,FWD_ACT_DATA_PKTS,ACTIVE_MEAN,ACTIVE_STD,ACTIVE_MAX,ACTIVE_MIN,IDLE_MEAN,IDLE_MAX,IDLE_MIN,LABEL
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Benign
1,1,0,1,1,1,1,1,1,0,1,...,1,1,0,0,0,0,0,0,0,Benign
2,0,0,2,2,2,2,2,2,0,2,...,2,2,0,0,0,0,0,0,0,Benign
3,0,0,3,1,3,3,1,3,0,3,...,3,1,0,0,0,0,0,0,0,Benign
4,0,0,4,0,4,4,3,4,0,4,...,2,2,0,0,0,0,0,0,0,Benign


In [16]:
print("Distribución final de clases:")
display(df[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Distribución final de clases:


,count
LABEL,
Benign,10630624
DDoS attacks-LOIC-HTTP,575364
DDOS attack-HOIC,198861
DoS attacks-Hulk,145199
Bot,144535
Infilteration,139775
SSH-Bruteforce,94048
DoS attacks-GoldenEye,41406
DoS attacks-Slowloris,9908


In [17]:
guardar_dataset_csv(
    df=df,
    nombre_archivo=NOMBRE_DATASET_LIMPIO,
    ruta=RUTA_SALIDA
)

print("Dataset limpio guardado correctamente.")
print(Path(RUTA_SALIDA).resolve() / NOMBRE_DATASET_LIMPIO)

Dataset limpio guardado correctamente.
/LUSTRE/home/inginf/u32902122/TFG/02_datasets/processed_analisis_estadistico/CIC18__cleanning__v2/CIC18__cleanning__v2.csv
